# Exploratory Data Analysis

In this notebook we explore the **Walmart Recruiting — Store Sales
Forecasting** dataset. Our goal is to understand the data well enough that
we can make informed feature engineering and modeling choices in the next
notebook.

We will:

1. Load the three source files (`stores`, `features`, `train`) and inspect
   their shape, columns, and types.
2. Check data quality — missing values, duplicates, and value ranges.
3. Profile the **target** (`Weekly_Sales`): distribution, outliers, and
   time trend.
4. Look at structure across **stores** and **departments**.
5. Investigate the impact of **holidays**, **temperature**, **fuel price**,
   and **markdowns**.
6. Wrap up with a list of takeaways that will guide modeling choices.


## Setup

We will use **`pandas`** for tabular work, **`numpy`** for numeric helpers,
and **`plotly express`** for interactive visualizations. Plotly is well
suited to this dataset because the time series spans almost three years and
we will want to zoom and hover to inspect specific weeks.


In [ ]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px

print("pandas version :", pd.__version__)
print("numpy version  :", np.__version__)
print("plotly version :", plotly.__version__)

## Load the Data

The dataset is split across three CSV files. We load them straight from the
mirrored copy on GitHub so the notebook is reproducible without any local
downloads.


### `stores.csv`

One row per Walmart store (45 stores total). Tells us the **type** of each
store (`A`, `B`, or `C`, which loosely correspond to large / medium /
small format) and its **size** in square feet.


In [ ]:
df_stores = pd.read_csv(
    "https://raw.githubusercontent.com/bdi593/datasets/refs/heads/main/walmart-store-sales-forecasting/stores.csv"
)
df_stores

### `features.csv`

One row per `(Store, Week)`. Contains contextual variables Walmart thinks
might explain sales: weather, fuel price, the consumer price index,
unemployment rate, an `IsHoliday` flag, and five anonymized markdown
columns capturing promotional spend.

> **Heads-up:** the `MarkDown1` … `MarkDown5` columns are only available
> after November 2011 and are missing (`NaN`) before that. We will deal
> with this when we engineer features.


In [ ]:
df_features = pd.read_csv(
    "https://raw.githubusercontent.com/bdi593/datasets/refs/heads/main/walmart-store-sales-forecasting/features.csv",
    parse_dates=["Date"],
)
df_features.head()

### `train.csv`

One row per `(Store, Department, Week)` — this is our **target series**.
The column `Weekly_Sales` is what we will eventually try to forecast.


In [ ]:
df_train = pd.read_csv(
    "https://raw.githubusercontent.com/bdi593/datasets/refs/heads/main/walmart-store-sales-forecasting/train.csv",
    parse_dates=["Date"],
)
df_train.head()

## Data Shape and Types

A quick sanity check: how big is each table, what types are the columns,
and how much of the data is missing?


In [ ]:
for name, df in [("stores", df_stores), ("features", df_features), ("train", df_train)]:
    print(f"--- {name} ---")
    print(f"  shape : {df.shape}")
    print(f"  date range: {df['Date'].min()} → {df['Date'].max()}" if "Date" in df.columns else "")
    print(df.dtypes)
    print()

### Missing Values

We expect the `MarkDown` columns in `features` to be largely missing
(they only start in late 2011). Anything else missing would be a surprise.


In [ ]:
df_features.isna().mean().sort_values(ascending=False).round(3)

In [ ]:
print("missing in stores :", df_stores.isna().sum().sum())
print("missing in train  :", df_train.isna().sum().sum())

As expected, the markdown columns are missing for the majority of weeks.
The `CPI` and `Unemployment` columns also have a small amount of missing
data toward the end of the series. The training table itself has no
missing values.


## Stores

Let's profile the 45 stores by **type** and **size** before diving into
sales.


In [ ]:
df_stores["Type"].value_counts()

In [ ]:
fig = px.box(
    df_stores,
    x="Type",
    y="Size",
    color="Type",
    points="all",
    title="Store size (sq ft) by store type",
)
fig.show()

**Takeaway.** Type `A` stores are the largest, type `C` are the smallest,
and type `B` overlaps the middle. Store *size* is not just a label — it is
a strong predictor of how much a store can sell, and we will keep both the
type and the raw size as features.


## Target Variable: `Weekly_Sales`

The thing we ultimately want to forecast. We will look at:

- the **distribution** of weekly sales across all `(Store, Dept, Week)`
  rows,
- whether there are any **negative or zero** sales (returns or closed
  departments),
- and the overall **time trend** at the chain level.


In [ ]:
df_train["Weekly_Sales"].describe()

A handful of rows have **negative** weekly sales — those are weeks where
returns exceeded purchases for that department. We will leave them in for
now but flag them as something to watch.


In [ ]:
neg = (df_train["Weekly_Sales"] < 0).sum()
print(f"rows with negative weekly sales: {neg:,} ({neg / len(df_train):.2%})")

In [ ]:
fig = px.histogram(
    df_train,
    x="Weekly_Sales",
    nbins=80,
    title="Distribution of Weekly_Sales (raw)",
)
fig.show()

The distribution is **highly right-skewed**: most department-weeks sit
under \$20k while a long tail stretches past \$500k. A log transform
gives a much more readable picture and is the kind of trick we may want
to apply when training certain models.


In [ ]:
positive = df_train.loc[df_train["Weekly_Sales"] > 0].copy()
positive["log_sales"] = np.log1p(positive["Weekly_Sales"])

fig = px.histogram(
    positive,
    x="log_sales",
    nbins=60,
    title="Distribution of log(Weekly_Sales + 1)",
)
fig.show()

### Total Weekly Sales over Time

Aggregating up to the **chain level** (sum over all stores and
departments) shows the overall demand pattern.


In [ ]:
weekly_total = (
    df_train.groupby("Date", as_index=False)["Weekly_Sales"].sum()
)

fig = px.line(
    weekly_total,
    x="Date",
    y="Weekly_Sales",
    title="Total Walmart weekly sales (all stores & departments)",
)
fig.show()

**Takeaway.** Two things jump out:

1. **Strong yearly seasonality** — every November/December has a massive
   spike driven by Thanksgiving, Black Friday, and Christmas.
2. **Weak overall trend** — outside of the holiday spikes the baseline
   level is fairly flat across the three years, with a mild dip in 2010.

Any forecasting model we build *must* capture this seasonality, either via
calendar features, lagged values, or a seasonal model component.


## Holidays

The dataset highlights four "official" holidays: **Super Bowl**, **Labor
Day**, **Thanksgiving**, and **Christmas**. The Kaggle competition
weights errors on holiday weeks 5× higher than non-holiday weeks, so it
pays to look at them carefully.


In [ ]:
holiday_summary = (
    df_train.groupby("IsHoliday")["Weekly_Sales"]
    .agg(["count", "mean", "median"])
    .round(2)
)
holiday_summary

In [ ]:
fig = px.box(
    df_train,
    x="IsHoliday",
    y="Weekly_Sales",
    title="Weekly_Sales: holiday vs. non-holiday weeks",
)
fig.update_yaxes(range=[-5_000, 100_000])  # zoom past the long tail
fig.show()

**Takeaway.** Holiday weeks have a noticeably higher mean and a much
heavier upper tail. Splitting `IsHoliday` apart further (e.g., labeling
which holiday) is a useful feature engineering idea for the modeling
notebook.


## Store and Department Breakdown

The chain-level view hides a lot of variation. Walmart sells very
different mixes of products in different departments, and store size
affects total volume.


In [ ]:
sales_by_store = (
    df_train.groupby("Store", as_index=False)["Weekly_Sales"].sum()
    .merge(df_stores, on="Store")
    .sort_values("Weekly_Sales", ascending=False)
)

fig = px.bar(
    sales_by_store,
    x="Store",
    y="Weekly_Sales",
    color="Type",
    title="Total sales by store, colored by store type",
)
fig.show()

As expected, large type-`A` stores dominate total sales. Store *type*
will likely be a useful categorical feature.


In [ ]:
sales_by_dept = (
    df_train.groupby("Dept", as_index=False)["Weekly_Sales"].sum()
    .sort_values("Weekly_Sales", ascending=False)
    .head(20)
)

fig = px.bar(
    sales_by_dept,
    x="Dept",
    y="Weekly_Sales",
    title="Top 20 departments by total sales",
)
fig.update_xaxes(type="category")
fig.show()

A handful of departments account for a large share of revenue. For
modeling we will want to forecast at the `(Store, Dept)` level rather than
the store level.


## Joining Sales with Features

To examine how weather, fuel price, and markdowns relate to sales we join
`train` with `features` on `(Store, Date)`.


In [ ]:
df = (
    df_train
    .merge(df_features, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(df_stores, on="Store", how="left")
)
print("merged shape:", df.shape)
df.head()

### Temperature vs. Sales

Average regional temperature could matter (think: heaters, snow shovels,
swimsuits). We aggregate to the weekly chain level so the picture isn't
dominated by store-level noise.


In [ ]:
weekly = (
    df.groupby("Date")
    .agg(
        Weekly_Sales=("Weekly_Sales", "sum"),
        Temperature=("Temperature", "mean"),
        Fuel_Price=("Fuel_Price", "mean"),
        CPI=("CPI", "mean"),
        Unemployment=("Unemployment", "mean"),
    )
    .reset_index()
)

fig = px.scatter(
    weekly,
    x="Temperature",
    y="Weekly_Sales",
    trendline="ols",
    title="Avg. weekly temperature vs. total weekly sales",
)
fig.show()

**Takeaway.** The relationship is weak at the chain level — total sales
are similar across most temperature ranges, with the biggest spikes
driven by *holidays*, not weather. At the **department** level the
relationship would likely be much stronger (e.g., garden vs. winter
apparel), but capturing that is a job for an interaction feature in the
model, not a single global slope.


### Fuel Price and Unemployment over Time

Both rise and fall with the broader economy. They are easy features to
include but are unlikely to *cause* short-term swings in weekly sales —
they are slow-moving context, not week-to-week shocks.


In [ ]:
fig = px.line(
    weekly.melt(id_vars="Date", value_vars=["Fuel_Price", "Unemployment", "CPI"]),
    x="Date",
    y="value",
    color="variable",
    facet_row="variable",
    title="Macro context over time",
    height=600,
)
fig.update_yaxes(matches=None)
fig.show()

### Markdowns

Markdowns are promotional spend by category. They are missing for most of
2010 and the first half of 2011, but become widely populated from late
2011 onward.


In [ ]:
markdown_cols = [f"MarkDown{i}" for i in range(1, 6)]

md_long = (
    df.groupby("Date")[markdown_cols].sum(min_count=1).reset_index()
    .melt(id_vars="Date", var_name="MarkDown", value_name="Spend")
)

fig = px.line(
    md_long,
    x="Date",
    y="Spend",
    color="MarkDown",
    title="Total markdown spend by week",
)
fig.show()

**Takeaway.** Markdowns spike around the major holidays, especially
Thanksgiving / Black Friday and Christmas. They could be useful features,
but their **missingness pattern** means we will need to be careful when
training a model that mixes pre-2011 and post-2011 weeks.


## A Single Series Up Close

Looking at one `(Store, Dept)` series gives us intuition for the
forecasting problem we are about to tackle.


In [ ]:
one_series = df_train.query("Store == 1 and Dept == 1").sort_values("Date")

fig = px.line(
    one_series,
    x="Date",
    y="Weekly_Sales",
    title="Store 1, Department 1 — weekly sales",
)
fig.show()

Even in a single department you can see clear yearly seasonality and a
few sharp holiday spikes. A good forecaster will need to learn both the
**level** of each `(Store, Dept)` series and its **seasonal pattern**.


## Key Takeaways

Things to carry into the modeling notebook:

1. **Forecast at the `(Store, Dept)` grain.** That is the natural unit of
   the data and where the variation lives.
2. **Seasonality is huge** — calendar features (week of year, month,
   holiday flags, weeks-to-next-holiday) will be essential.
3. **Holidays drive the biggest swings.** Encoding *which* holiday is
   coming up matters more than just whether the week is a holiday.
4. **Store type and size** are useful static features.
5. **Markdown columns are missing for ~half the period** — handle their
   `NaN`s explicitly (impute, indicator flag, or restrict to post-2011
   training).
6. **Weekly sales are right-skewed and occasionally negative.** Consider a
   log or `log1p` transform for linear-style models, and decide how to
   treat the negative-sales rows (drop, clip to 0, or model directly).
7. **Use a time-based train/test split** — random shuffles will leak
   future weeks into the training set.
